In [10]:
import pytest
from rorycommon import Common as RoryCommon
import os 
import numpy as np
from mictlanx import AsyncClient
from mictlanx.utils import Utils
from concurrent.futures import ProcessPoolExecutor
from rory.core.security.dataowner import DataOwner
from rory.core.security.cryptosystem.liu import Liu
from xolo.utils.utils import Utils as XoloUtils
import hashlib as H
MICTLANX_CLIENT_ID           = os.environ.get("MICTLANX_CLIENT_ID","{}_mictlanx".format("rory-common"))
MICTLANX_TIMEOUT             = int(os.environ.get("MICTLANX_TIMEOUT",3600))
MICTLANX_API_VERSION         = int(os.environ.get("MICTLANX_API_VERSION","3"))
MICTLANX_ROUTERS             = os.environ.get("MICTLANX_ROUTERS", "mictlanx-router-0:localhost:60666") #mictlanx-peer-2:localhost:7002")
MICTLANX_DEBUG               = bool(int(os.environ.get("MICTLANX_DEBUG",0)))
MICTLANX_DAEMON              = bool(int(os.environ.get("MICTLANX_DAEMON",1)))
MICTLANX_SHOW_METRICS        = bool(int(os.environ.get("MICTLANX_SHOW_METRICS",0)))
MICTLANX_DISABLED_LOG        = bool(int(os.environ.get("MICTLANX_DISABLED_LOG",0)))
MICTLANX_MAX_WORKERS         = int(os.environ.get("MICTLANX_MAX_WORKERS","12"))
MICTLANX_CLIENT_LB_ALGORITHM = os.environ.get("MICTLANX_CLIENT_LB_ALGORITHM","2CHOICES_UF")
MICTLANX_BUCKET_ID           = os.environ.get("MICTLANX_BUCKET_ID","rory") 
MICTLANX_OUTPUT_PATH         = os.environ.get("MICTLANX_OUTPUT_PATH","/rory/mictlanx")

client = AsyncClient(
    client_id=MICTLANX_CLIENT_ID,
    capacity_storage="200mb",
    debug=False,
    eviction_policy="LRU",
    max_workers= MICTLANX_MAX_WORKERS,
    routers=list(Utils.routers_from_str(routers_str=MICTLANX_ROUTERS,protocol="https")),
    verify=False
)
dataowner = DataOwner(
    liu_scheme= Liu(
        _round=True,
        decimals=10
    ),
)
scale_factor = 10**9

In [18]:
key = "encryptedskmeansx"
bucket_id = "rory"
pmt = await RoryCommon.read_numpy_from(
    path="/rory/source/clusteringc0r10a5k20.npy",
    extension="npy"
)
pmt = pmt.unwrap()
# pmt = (pmt * scale_factor).astype(np.int64)
# print(pmt.dtype)
# np.random.seed(10)

n = pmt.shape[0]*pmt.shape[1]*dataowner.m
num_chunks = 2
emt = RoryCommon.segment_and_encrypt_liu_with_executor(
    executor= ProcessPoolExecutor(max_workers=num_chunks),
    dataowner=dataowner,
    key=key,
    n=n,
    np_random=True,
    num_chunks=num_chunks,
    plaintext_matrix=pmt
)

In [21]:
h = H.sha256()
for c in emt:
    print(c.chunk_id,c.checksum)
    h.update(c.data)
    print(c.to_ndarray())
print("GLOBAL_H",h.hexdigest())

encryptedskmeansx_0 f03a892d905e18f8fbfa4815809e3468ddb8dd8dc2e11dff38714b46057f9004
Some(array([[[ 2.87696007e+07,  9.00294459e+06,  3.25224908e+04],
        [ 2.78458072e+07,  8.71390363e+06,  3.25224908e+04],
        [ 6.64201244e+07,  2.07832218e+07,  3.25224908e+04],
        [-6.72694887e+07, -2.10462283e+07,  3.25224908e+04],
        [-5.96742918e+07, -1.86698065e+07,  3.25224908e+04]],

       [[-8.66631671e+06, -2.71018558e+06,  3.25224908e+04],
        [-5.62016239e+07, -1.75832615e+07,  3.25224908e+04],
        [-7.39669831e+07, -2.31417726e+07,  3.25224908e+04],
        [ 5.81662937e+07,  1.82007235e+07,  3.25224908e+04],
        [-5.38173999e+07, -1.68372740e+07,  3.25224908e+04]],

       [[-7.45049162e+06, -2.32977237e+06,  3.25224908e+04],
        [-5.79225293e+07, -1.81217066e+07,  3.25224908e+04],
        [-7.32761346e+07, -2.29256166e+07,  3.25224908e+04],
        [ 5.97336030e+07,  1.86911108e+07,  3.25224908e+04],
        [-5.64624034e+07, -1.76648554e+07,  3.252249

In [15]:

checksum,_ = XoloUtils.sha256_stream(emt.to_generator())
checksum1= XoloUtils.sha256(emt.to_bytes())
print("C1",checksum)
print("C2",checksum1)
put_res = await RoryCommon.put_chunks(
    client=client,
    key=key,
    bucket_id=bucket_id,
    chunks=emt,
    tags={
        "full_shape":str((pmt.shape[0],pmt.shape[1],dataowner.m)),
        "full_dtype":str(pmt.dtype)
    }
) 
print(put_res)


C1 cc4314cd0d715600784e8597bcd31f3606b2564900db3788140b00d6b60c822b
C2 cc4314cd0d715600784e8597bcd31f3606b2564900db3788140b00d6b60c822b
PUT_CHUNKS_cHJECKSUM cc4314cd0d715600784e8597bcd31f3606b2564900db3788140b00d6b60c822b cc4314cd0d715600784e8597bcd31f3606b2564900db3788140b00d6b60c822b
Ok(True)


In [16]:
key = "encryptedskmeansx"
bucket_id = "rory"
x = await RoryCommon.get_and_merge(
    client = client, 
    key = key, 
    bucket_id = bucket_id
)
x


100%|██████████| 2/2 [00:00<00:00, 17.00it/s, chunk=1, resp_time=0.12s]

X float64
X float64
CHECKSUM b2a6a80d382f6c26f28c3b2f443cf25e90abca66e1ae852ef5ac8f3d2401c48d


array([[[ 7.83406158e+06,  1.05889234e+07,  2.52142633e+04],
        [ 7.61704566e+06,  1.02954795e+07,  2.52142633e+04],
        [ 1.82508257e+07,  2.46742316e+07,  2.52142633e+04],
        [-1.84248647e+07, -2.49177909e+07,  2.52142633e+04],
        [-1.62547055e+07, -2.19833517e+07,  2.52142633e+04]],

       [[-2.36568666e+06, -3.20294082e+06,  2.52142633e+04],
        [-1.53866419e+07, -2.08095760e+07,  2.52142633e+04],
        [-2.01609921e+07, -2.72653423e+07,  2.52142633e+04],
        [ 1.58636506e+07,  2.14463485e+07,  2.52142633e+04],
        [-1.47355941e+07, -1.99292443e+07,  2.52142633e+04]],

       [[-1.93165482e+06, -2.61605298e+06,  2.52142633e+04],
        [-1.58206737e+07, -2.13964639e+07,  2.52142633e+04],
        [-1.99439762e+07, -2.69718983e+07,  2.52142633e+04],
        [ 1.62976825e+07,  2.20332363e+07,  2.52142633e+04],
        [-1.53866419e+07, -2.08095760e+07,  2.52142633e+04]],

       [[ 6.96599790e+06,  9.41514774e+06,  2.52142633e+04],
        [ 8.268093

In [77]:
xx = dataowner.liu_scheme.decryptMatrix(
    ciphertext_matrix=x,
    secret_key= dataowner.liu_scheme.sk,
)
xx.matrix


array([[ 0.10657093,  0.10361449,  0.24847985, -0.25115781, -0.22159346],
       [-0.03238156, -0.20976771, -0.2748093 ,  0.21595905, -0.2008984 ],
       [-0.02646869, -0.21568058, -0.27185287,  0.22187193, -0.20976771],
       [ 0.09474518,  0.1124838 ,  0.26621846, -0.24820138, -0.20976771],
       [-0.05012017, -0.21272415, -0.26889643,  0.21595905, -0.2008984 ],
       [ 0.10361449,  0.10361449,  0.26326203, -0.24820138, -0.20976771],
       [ 0.10657093,  0.10065806,  0.25734916, -0.24820138, -0.22454989],
       [ 0.10657093,  0.10065806,  0.26326203, -0.24820138, -0.22159346],
       [-0.02942512, -0.22159346, -0.28072217,  0.22187193, -0.21863702],
       [-0.03238156, -0.21863702, -0.27185287,  0.2277848 , -0.21863702]])

In [17]:
pmt

array([[ 0.3619735 ,  0.35034416,  0.83594393, -0.84703195, -0.75141844],
       [-0.10929532, -0.70770214, -0.93134457,  0.73203909, -0.6776879 ],
       [-0.09398968, -0.72936606, -0.9226477 ,  0.75176945, -0.71098501],
       [ 0.32198933,  0.38248296,  0.89773652, -0.83899041, -0.71271491],
       [-0.16510177, -0.71946749, -0.91383501,  0.72914892, -0.68200372],
       [ 0.34840901,  0.34507709,  0.88647564, -0.83779848, -0.71387043],
       [ 0.35502391,  0.33770914,  0.87477984, -0.83804798, -0.75933998],
       [ 0.36190371,  0.34341381,  0.89085246, -0.8380352 , -0.7479102 ],
       [-0.10458927, -0.75206596, -0.94830775,  0.7549219 , -0.74418975],
       [-0.10919271, -0.74255267, -0.91918663,  0.76583693, -0.74239348]])